# GestureLens — MobileNetV2 fine-tuning on HaGRID

Kaggle training notebook. This mirrors `training/train.py` but is set up to run
against a Kaggle Dataset input rather than a local directory.

Steps:
1. Attach the HaGRID dataset (or your class subset) as a Kaggle Dataset input
2. Build train/val datasets
3. Fine-tune MobileNetV2
4. Save the model for downstream TFLite/TF.js conversion

In [ ]:
import tensorflow as tf
print(tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
# [TODO] Point this at the Kaggle input path, e.g. /kaggle/input/hagrid-classification-512p
DATA_DIR = '/kaggle/input/hagrid-dataset'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training', seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation', seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE)
class_names = train_ds.class_names
print(class_names)

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1./255)
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
])

def prep_train(x, y):
    x = augmentation(x)
    x = normalization_layer(x)
    return x, y

def prep_val(x, y):
    return normalization_layer(x), y

train_ds = train_ds.map(prep_train).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(prep_val).prefetch(tf.data.AUTOTUNE)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

In [ ]:
model.save('/kaggle/working/gesturelens.h5')
with open('/kaggle/working/class_names.txt', 'w') as f:
    f.write('\n'.join(class_names))
print('Saved model + class names to /kaggle/working/')